# 卫星图像分析平台 - ONNX → TF.js 转换笔记本

**用途**：在 Google Colab 上把已经训练好的 ONNX 模型转换成 TensorFlow.js 图模型，方便前端 `tfjs` 加载。

**前置条件**：
- 你已经按照 `.trae/specs/satellite-image-full/tasks.md` Task 16 跑过 `python scripts/convert_to_onnx.py --weights models/checkpoints/best.pt --output models/web_model/best.onnx`
- 最佳实践：把 `best.onnx` 推送到 GitHub Release 或一个公开 URL，再在 Cell 3 把 `<user>/<repo>` 替换成你的真实 GitHub 用户名和仓库名

**产物**：
- `web_model_tfjs/model.json`
- `web_model_tfjs/group1-shard1of*.bin`（按张量分片）
- 自动打包成 `web_model_tfjs.zip` 供下载

## Step 1: 安装依赖

TF2.16+ 需要 `tensorflow-io-gcs-filesystem`，否则 SavedModel 写入会报错。

In [ ]:
!pip install -q onnx==1.17.0 onnx2tf tensorflow==2.16.1 tensorflow-io-gcs-filesystem==0.37.1 tensorflowjs==4.20.0

## Step 2: 选择输入来源（二选一）

**方式 A（推荐）**：从 GitHub Release / LFS 公开 URL 下载。  
**方式 B**：从 Colab 侧边栏文件面板上传 `best.onnx`。

下面的 Cell 3 默认走方式 A；如果你走方式 B，请**跳过 Cell 3**，把上传的文件重命名/移动到 `/content/best.onnx`。

In [ ]:
# ==================== 方式 A：从 GitHub 下载 ONNX ====================
# ↓↓↓ 把 <user> 和 <repo> 替换成你的真实 GitHub 用户名/仓库名 ↓↓↓
GITHUB_USER = "<your-github-user>"
GITHUB_REPO = "<your-repo>"
ONNX_PATH   = "main"            # 分支名
ONNX_FILE   = "models/web_model/best.onnx"

RAW_URL = f"https://raw.githubusercontent.com/{GITHUB_USER}/{GITHUB_REPO}/{ONNX_PATH}/{ONNX_FILE}"
print(f"[INFO] 下载: {RAW_URL}")

!wget -q -O best.onnx "$RAW_URL"
!ls -lh best.onnx

## Step 3: ONNX → TensorFlow SavedModel

使用 [`onnx2tf`](https://github.com/PINTO0309/onnx2tf)，它对 `dynamic_axes`、自定义 op 兼容性较好。

如果看到 `DequantizeLinear` / `DynamicQuantizeLinear` 相关报错，加 `-osd` 关闭 shape 推断重跑。

In [ ]:
!rm -rf tf_saved/
!onnx2tf -i best.onnx -o tf_saved/ -osd 2>&1 | tail -n 60

## Step 4: TensorFlow SavedModel → TF.js Graph Model

产出 `web_model_tfjs/model.json` + `group*-shard*.bin`。  
前端用 `tf.loadGraphModel('/static/web_model_tfjs/model.json')` 加载即可。

In [ ]:
!rm -rf web_model_tfjs/
!tensorflowjs_converter \
    --input_format=tf_saved_model \
    --output_format=tfjs_graph_model \
    --signature_name=serving_default \
    --weight_shard_size_bytes=4194308 \
    tf_saved/ web_model_tfjs/

!ls -lh web_model_tfjs/

## Step 5: 打包并下载

运行后会触发浏览器下载 `web_model_tfjs.zip`。

In [ ]:
!zip -r -q web_model_tfjs.zip web_model_tfjs/
!ls -lh web_model_tfjs.zip

from google.colab import files
files.download('web_model_tfjs.zip')

## 验证（可选）

把下载的 `web_model_tfjs/` 解压到 `frontend/public/web_model_tfjs/`，然后在浏览器里跑：

```js
import * as tf from '@tensorflow/tfjs'
const m = await tf.loadGraphModel('/web_model_tfjs/model.json')
m.summary()        // 应该看到卷积/EfficientNet 模块
const x = tf.randomNormal([1, 384, 384, 3])
const y = m.predict(x)
y.print()          // shape: [1, 10]
```

如果 `y.shape === [1, 10]`，转换链路完整闭环 ✅